In [28]:
import pandas as pd

df = pd.read_csv('a_hostel_walking.csv')

In [29]:
df.head()

,id,time,amplitude1_1,amplitude2_1,amplitude3_1,amplitude4_1,amplitude5_1,amplitude6_1,amplitude7_1,amplitude8_1,...,phase58_2,phase59_2,phase60_2,phase61_2,phase62_2,phase63_2,phase64_2,rssi_1,rssi_2,activity
0,39582,23:34:48,115.277925,4.0,40.049969,36.221541,34.525353,31.575307,26.305893,21.023796,...,0.000000,0.000000,-0.049958,-0.047583,-0.086738,-0.129703,-0.179853,-50,-52,0
1,39583,23:34:49,115.277925,4.0,38.275318,35.805028,32.802439,29.206164,24.207437,20.808652,...,-1.844964,-1.892547,-1.919567,-1.946067,-1.946067,-1.935998,-1.919567,-51,-52,0
2,39584,23:34:49,115.277925,4.0,41.677332,38.483763,36.769553,33.241540,27.313001,21.954498,...,2.553590,2.512796,2.483704,2.455863,2.403778,2.425849,2.403778,-51,-52,0
3,39585,23:34:49,115.277925,4.0,42.801869,41.036569,38.897301,35.383612,29.000000,21.931712,...,1.525373,1.529154,1.570796,1.533776,1.536327,1.570796,1.603043,-51,-53,0
4,39586,23:34:49,115.277925,4.0,43.278170,38.897301,36.069378,29.732137,24.041631,19.104973,...,-2.568627,-2.427502,-2.373142,-2.532903,-2.709185,-2.714965,-2.662073,-51,-52,0


In [30]:
import joblib

model = joblib.load('./models/gmm_model_1.pkl')

In [31]:
import joblib

pca = joblib.load('./models/pca_model.joblib')

In [32]:
import pywt
import numpy as np

wavelet_name = 'gaus1'

scales = np.arange(1, 128)	

In [33]:
import numpy as np

def getWavelet(data):
	X = data.drop(columns = ['activity', 'id', 'time'])
	pca_X = pd.DataFrame(pca.transform(X))
	coefficients, frequencies = pywt.cwt(pca_X.iloc[: , 0], scales, wavelet_name)
	wavelet_variances = np.sum(coefficients**2, axis=0)
	return wavelet_variances



In [34]:
def get25(group):
	fraction = len(group) /25
	forPredict = []
	for i in range(1,26):
		index = int(i * fraction)
		forPredict.append(group.iloc[index-1])

	return forPredict

In [35]:
grouped_data = df.groupby('time')
i = 0
wavelet_list = np.zeros((1, 25))

for time, group in grouped_data:
	if len(group) > 25:
		forPredict = get25(group)
		wavelet = getWavelet(pd.DataFrame(forPredict))
		wavelet_list = np.vstack([wavelet_list, wavelet])

wavelet_list = np.delete(wavelet_list, np.s_[:1], axis=0)

wavelet_list = pd.DataFrame(wavelet_list)

wavelet_list.columns = [str(i) for i in range(0, 25)]

model.fit(wavelet_list)



GaussianMixture(n_components=3)

In [36]:
joblib.dump(model, './models/gmm_model_1.pkl')

['./models/gmm_model_1.pkl']